# Build DiD Monthly Panel — China WTO Accession

This notebook constructs a Difference-in-Differences (DiD) panel to estimate the impact of China's WTO entry (January 2002) on U.S. imports from China. Treatment is based on pre-policy NTR-gap exposure at the HS8 product level.

## Segment 1: Load & Inspect Data

We start from `panel_hts10_monthly.csv` — a pre-cleaned monthly trade panel at the HTS10 product level. This file was built from raw USITC DataWeb imports data. We also need `tar_val.dta`, which contains tariff/NTR-gap data from Pierce & Schott (2016, AER).

**Why these files?**
- The trade panel gives us monthly import values and quantities by product (HTS10 = 10-digit Harmonized Tariff Schedule code)
- The tariff file gives us the "NTR gap" — the difference between the high (non-NTR) tariff and the low (NTR) tariff for each product. Products with a bigger gap had more to gain from permanent NTR status, making them more "treated" by China's WTO entry.

In [1]:
import pandas as pd
import numpy as np
import os

# File paths (relative to notebook location in Code/)
panel_path = "../transformed_data/panel_hts10_monthly.csv"
tariff_path = "../transformed_data/data_files_aer_2013/tar_val.dta"

# Verify files exist
for name, path in [("Trade panel", panel_path), ("Tariff file", tariff_path)]:
    exists = os.path.exists(path)
    size = f"{os.path.getsize(path) / 1e6:.1f} MB" if exists else "NOT FOUND"
    print(f"{name}: {path} — {size}")

Trade panel: ../transformed_data/panel_hts10_monthly.csv — 119.2 MB
Tariff file: ../transformed_data/data_files_aer_2013/tar_val.dta — 5.9 MB


### Load the trade panel and inspect its structure

We need to understand the columns before doing any transformations. Key things to look for:
- What does each row represent?
- What are the product code and date columns?
- What are the value/quantity columns?
- Are there any obvious data type issues (e.g., product codes read as numbers, losing leading zeros)?

In [2]:
# Load trade panel
panel = pd.read_csv(panel_path, dtype={"hts10": str})

print(f"Shape: {panel.shape[0]:,} rows × {panel.shape[1]} columns")
print(f"\nColumns:\n{list(panel.columns)}")
print(f"\nData types:\n{panel.dtypes}")
print(f"\nFirst 5 rows:")
panel.head()

Shape: 1,251,312 rows × 11 columns

Columns:
['Country', 'Year', 'Month', 'hts10', 'customs_value', 'first_unit_qty', 'date', 'unit_value', 'ln_customs_value', 'ln_first_unit_qty', 'ln_unit_value']

Data types:
Country                  str
Year                   int64
Month                    str
hts10                    str
customs_value          int64
first_unit_qty         int64
date                     str
unit_value           float64
ln_customs_value     float64
ln_first_unit_qty    float64
ln_unit_value        float64
dtype: object

First 5 rows:


,Country,Year,Month,hts10,customs_value,first_unit_qty,date,unit_value,ln_customs_value,ln_first_unit_qty,ln_unit_value
0,China,1996,January,0101110020,0,0,1996-01-01,NaN,0.000000,0.00000,NaN
1,China,1996,February,0101110020,0,0,1996-02-01,NaN,0.000000,0.00000,NaN
2,China,1996,March,0101110020,3150,6,1996-03-01,525.0,8.055475,1.94591,6.263398
3,China,1996,April,0101110020,0,0,1996-04-01,NaN,0.000000,0.00000,NaN
4,China,1996,May,0101110020,0,0,1996-05-01,NaN,0.000000,0.00000,NaN


In [3]:
# Check: are HTS10 codes proper 10-digit strings?
print("Sample hts10 values:", panel["hts10"].head(10).tolist())
print(f"\nAll 10 digits? {(panel['hts10'].str.len() == 10).all()}")
print(f"Any NaN hts10? {panel['hts10'].isna().sum()}")

# Check countries present
print(f"\nUnique countries: {panel['Country'].nunique()}")
print(f"Country values: {panel['Country'].unique()[:10]}")

# Check date range
print(f"\nDate range: {panel['date'].min()} to {panel['date'].max()}")
print(f"Unique dates: {panel['date'].nunique()}")

Sample hts10 values: ['0101110020', '0101110020', '0101110020', '0101110020', '0101110020', '0101110020', '0101110020', '0101110020', '0101110020', '0101110020']

All 10 digits? True
Any NaN hts10? 0

Unique countries: 1
Country values: <StringArray>
['China']
Length: 1, dtype: str

Date range: 1996-01-01 to 2005-12-01
Unique dates: 120


## Segment 2: Clean & Prepare for HS8 Aggregation

Our goal is to go from HTS10 (10-digit product codes) to HS8 (8-digit). Why?
- The tariff data (`tar_val.dta`) is at the HS8 level
- Multiple HTS10 codes map to the same HS8 (the last 2 digits are US-specific detail)
- We need to match granularity for the DiD merge

Steps:
1. Parse the `date` column to datetime
2. Drop the 11 known problematic HTS10 codes (products that switched quantity units over time — mixing units would corrupt our quantity analysis)
3. Create the `hs8` column (first 8 digits of `hts10`)

In [4]:
# Parse date to datetime
panel["date"] = pd.to_datetime(panel["date"])

# Drop known HTS10 codes with quantity-description switches
# These products changed how they measure quantity (e.g., from kg to units),
# so summing their quantities over time would be meaningless
problem_hts10 = [
    "2607000020", "2711290060", "6111206050", "6111305050", "6111905050",
    "7115900530", "7115900590", "7115903000", "7115904000", "7115906000",
    "9001300000"
]

rows_before = len(panel)
panel = panel[~panel["hts10"].isin(problem_hts10)].copy()
rows_after = len(panel)

print(f"Dropped {rows_before - rows_after:,} rows from {len(problem_hts10)} problematic HTS10 codes")
print(f"Remaining: {rows_after:,} rows")

# Create HS8 (first 8 digits)
panel["hs8"] = panel["hts10"].str[:8]

print(f"\nUnique HTS10 codes: {panel['hts10'].nunique():,}")
print(f"Unique HS8 codes:  {panel['hs8'].nunique():,}")
print(f"(Multiple HTS10s can map to one HS8)")

Dropped 720 rows from 11 problematic HTS10 codes
Remaining: 1,250,592 rows

Unique HTS10 codes: 16,276
Unique HS8 codes:  9,711
(Multiple HTS10s can map to one HS8)


In [5]:
# Quick sanity check: how many HTS10 codes per HS8?
hts10_per_hs8 = panel.groupby("hs8")["hts10"].nunique()
print("HTS10 codes per HS8:")
print(hts10_per_hs8.describe())
print(f"\nExample — HS8 '01011100' contains HTS10s: {panel[panel['hs8'] == '01011100']['hts10'].unique()}")

HTS10 codes per HS8:
count    9711.000000
mean        1.676037
std         1.751822
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        35.000000
Name: hts10, dtype: float64

Example — HS8 '01011100' contains HTS10s: <StringArray>
['0101110020']
Length: 1, dtype: str


## Segment 3: Aggregate to HS8-Month Panel

**Teaching guide — General Task 3 / Pseudocode STEP C**

Multiple HTS10 codes can map to one HS8. We aggregate up to (hs8, date) because our tariff treatment variable is at the HS8 level.

Rules:
- **Customs value**: always sum across HTS10s within (hs8, date)
- **Quantity**: sum across HTS10s (unit-switching codes were already dropped in Segment 2)
- **Unit value**: recompute as customs_value / quantity after aggregation

Note: The R `.qmd` also tracks `unit_consistent` (whether all HTS10s in an HS8-month share the same quantity unit). Our input panel lacks the `quantity_description` column, so we skip that check — the 11 problematic codes were already removed.

In [ ]:
# Aggregate HTS10 → HS8 monthly panel
panel_hs8 = (
    panel
    .groupby(["hs8", "date"], as_index=False)
    .agg(
        customs_value=("customs_value", "sum"),
        first_unit_qty_hs8=("first_unit_qty", "sum"),
        n_hts10_in_hs8=("hts10", "nunique"),
        year=("Year", "first")
    )
)

# Recompute unit_value at HS8 level
panel_hs8["unit_value"] = np.where(
    panel_hs8["first_unit_qty_hs8"] > 0,
    panel_hs8["customs_value"] / panel_hs8["first_unit_qty_hs8"],
    np.nan
)

print(f"HS8-month panel: {len(panel_hs8):,} rows")
print(f"Unique HS8 codes: {panel_hs8['hs8'].nunique():,}")
print(f"Date range: {panel_hs8['date'].min()} to {panel_hs8['date'].max()}")

In [ ]:
# Duplicate-key check: every (hs8, date) should appear exactly once
dup_check = panel_hs8.groupby(["hs8", "date"]).size()
n_dups = (dup_check > 1).sum()
print(f"Duplicate (hs8, date) keys: {n_dups}")

# Summary stats
print(f"\ncustoms_value summary:")
print(panel_hs8["customs_value"].describe())

print(f"\nunit_value summary (non-null):")
print(panel_hs8["unit_value"].dropna().describe())

print(f"\nRows with zero customs_value: {(panel_hs8['customs_value'] == 0).sum():,}")
print(f"Rows with missing unit_value: {panel_hs8['unit_value'].isna().sum():,}")